In [ ]:
import os
import pickle
import sys
from datetime import datetime

from ipywidgets import widgets
from matplotlib.lines import Line2D
from torch.optim import Adam
from tqdm import tqdm

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from src import *

plt.rcParams.update({'font.size': 18})

In [ ]:
df_x_1, df_y_1 = read_dataset(stage='classification', cheat=True)
df_x_2, df_y_2 = read_dataset(stage='classification', cheat=True, discarded=True)
df_x_1['domain'] = 1
df_x_2['domain'] = 2
df_x = pd.concat([df_x_1, df_x_2], ignore_index=True)
df_y = pd.concat([df_y_1, df_y_2], ignore_index=True)

In [ ]:
def get_title(model, domains, epochs, times, lr, is_file=False):
    additional = f'grid={model.grid_size}' if type(model) in [PyKAN, EfficientKAN] else ''
    return f'{model.__class__.__name__}_layers=[{"-".join(map(str, model.layers))}]_epochs={epochs}_times={times}_lr={lr}_{additional}_{datetime.now().ctime().replace(":", "-")}' if is_file else f'{model.__class__.__name__} Epochs={epochs} Times={times} Lr={lr}'

In [ ]:
def idl_train(model: PHMNetwork, domains: list[int], epochs: int, times: int, lr=1e-3, device='cpu') -> tuple[
    list, list, list]:
    model.reset()
    train_losses, valid_losses, test_metrics = [], [], []
    for i, domain in enumerate(tqdm(domains * times)):
        trainset, validset, testset, _ = split_dataset(df_x, df_y['faulty'], 0.5, standardize_y=False,
                                                       group_by='domain', device=device)
        min_length = min(trainset[1][0].shape[0], trainset[2][0].shape[0])
        for d in domains:
            trainset[d] = (trainset[d][0][:min_length], trainset[d][1][:min_length])

        def test():
            metrics = model.test(testset, batch_size=2048, silent=True)
            test_metrics[i].append(metrics)

        test_metrics.append([])
        train_loss, valid_loss = (
            model.fit(trainset[domain], validset[domain], Adam(model.parameters(), lr=lr), epochs=epochs,
                      batch_size=2048, prefix=f'IDL_{domain}', callback=test, silent=True))
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        model.save(f'IDL_{model.__class__.__name__}_{domain}')

    # Plot graph
    plot((train_losses, valid_losses, test_metrics), title=get_title(model, domains, epochs, times, lr))

    # Save to file
    with open(f'results/{get_title(model, domains, epochs, times, lr, is_file=True)}', 'wb') as f:
        pickle.dump((train_losses, valid_losses, test_metrics), f)
    return train_losses, valid_losses, test_metrics

In [ ]:
def plot(losses: tuple[list, list, list], train=False, valid=False, title=None, scale=1, ax=None,
         standalone=True, label=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(20 * scale, 10 * scale))
        ax.set_xticks(range(0, len(losses[0]) * len(losses[0][0]) + 1, len(losses[0][0])))
        for i, test_loss in enumerate(losses[0]):
            ax.add_line(
                Line2D([(i + 1) * len(test_loss)] * 2, [0, 99], linestyle='--', color='black', linewidth=1, alpha=0.5))
            ax.fill_between(range(i * len(test_loss), (i + 1) * len(test_loss) + 1), -1, 2, alpha=0.25,
                            color='tab:green' if i % 2 == 0 else 'tab:red',
                            label=None if i > 1 else 'np > ng' if i % 2 == 0 else 'np < ng')
        ax.set_ylim([-1, 0] if min(losses[0][-1]) < 0 else [0, 2])
    if train:
        ax.plot([x for xs in losses[0] for x in xs], color='tab:orange', label='train')
    if valid:
        ax.plot([x for xs in losses[1] for x in xs], color='tab:green', label='valid')
    ax.plot([x['test_loss'] for xs in losses[2] for x in xs], color='tab:blue' if standalone else None,
            label='test' if label is None else label, linewidth=3)

    if title: ax.set_title(title)
    if standalone:
        ax.legend()
        plt.show()
    return ax

In [ ]:
def visualizer_gui():
    output = widgets.Output()
    selected_files = []
    buttons = {}
    losses = {}

    def select_file(file, deselect_all=False):
        with output:
            output.clear_output()
            if deselect_all:
                selected_files.clear()
            else:
                if file in selected_files:
                    selected_files.remove(file)
                else:
                    selected_files.append(file)

            # Set button styles
            for f, button in buttons.items():
                button.button_style = 'primary' if f in selected_files else ''

            # Plot
            ax = None
            for selected_file in selected_files:
                ax = plot(losses[selected_file], title=file, train=False, ax=ax, standalone=False,
                          label=' '.join(selected_file.split('_')[:-1]))
            ax.legend()
            plt.show()

    files = os.listdir('results/')
    files = sorted(files, key=lambda f: os.path.getctime(os.path.join('results/', f)), reverse=False)

    for file in files:
        with open(f'results/{file}', 'rb') as f:
            losses[file] = pickle.load(f)

        btn = widgets.Button(
            description=' ——— '.join(file.split('_')[:-1]),
            layout=widgets.Layout(width='auto'),
            style={'font_size': '20px', 'text_decoration': 'underline' if min(losses[file][0][-1]) < 0 else ''},
            button_style='',
        )
        btn.on_click(lambda _, f=file: select_file(f))
        buttons[file] = btn
    clear = widgets.Button(
        description='Clear',
        layout=widgets.Layout(width='auto'),
        style={'font_size': '20px', 'font_weight': ''},
        button_style='warning',
    )
    clear.on_click(lambda _: select_file(None, deselect_all=True))
    spacer = widgets.HTML(value="<div style='margin-top:30px;'></div>")
    display(widgets.VBox([*buttons.values(), clear, spacer, output]))

In [ ]:
USE_NATIVE_LOSS = False
DOMAINS = [1, 2]
EPOCHS = 5
TIMES = 8
LR = 1e-2

# KAN

In [ ]:
for hidden in [[2]]:
    for grid in [10]:
        effKAN = EfficientKAN([len(df_x_1.columns) - 1, *hidden, 1], 'classification', grid_size=grid,
                              use_native_loss=USE_NATIVE_LOSS, continual_learning=True, device='cpu')
        idl_train(effKAN, DOMAINS, epochs=EPOCHS, times=TIMES, lr=LR, device='cpu')

# MLP

In [ ]:
for params in [8]:
    mlp = MLP([len(df_x_1.columns) - 1, params, params, 1], 'classification',
              use_native_loss=USE_NATIVE_LOSS, device='cpu')
    idl_train(mlp, DOMAINS, epochs=EPOCHS, times=TIMES, lr=LR)

In [ ]:
visualizer_gui()